In [2]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
import json

def show_response(response):
    """Messageオブジェクトをテキストで見やすく表示"""

    # メタ情報
    print(f"{'─'*60}")
    print(f"  Model      : {response.model}")
    print(f"  Stop reason: {response.stop_reason}")
    print(f"  Tokens     : input={response.usage.input_tokens}, output={response.usage.output_tokens}")
    print(f"{'─'*60}")

    # コンテンツブロック
    for i, block in enumerate(response.content):
        if block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"\n[Id    : {block.id}]")
            print(f"[Tool  : {block.name}]")
            print(json.dumps(block.input, indent=2, ensure_ascii=False))

    print(f"{'─'*60}")

### 火災リスクのプロンプト

1. **住居の特定**：以下の点に注目して、敷地内の主要な住居を特定してください：
	- 最も大きな屋根付き構造物
	- 一般的な住宅の特徴（車道との接続、規則的な形状）
	- 他の構造物との区別（ガレージ、物置、プール）

2. **樹木の張り出し分析**：主要な住居付近のすべての樹木を調査してください：
	- 樹冠が屋根のいずれかの部分に直接張り出している樹木を特定する
	- 張り出した枝で覆われている屋根の割合を推定する（0〜25%、25〜50%、50〜75%、75%以上）
	- 特に張り出しが密集している箇所を記録する

3. **火災リスク評価**：張り出している樹木について、以下を評価してください：
	- 山火事への脆弱性（飛び火の着火点、建物への連続的な燃料経路）
	- 煙突、通気口、その他の屋根の開口部が見える場合はその近接度
	- 枝が野生植生と建物の間の「橋渡し」となっている箇所

4. **防御空間の特定**：敷地全体の植生構造を評価してください：
	- 樹木が連続した樹冠を形成し、住宅の上または付近を覆っているかを確認する
	- 明らかな燃料はしご（地面から樹木、屋根へと火を運ぶ可能性のある植生）を記録する

5. **火災リスク評価レーティング**：分析に基づき、火災リスク評価を1〜4で評価してください：

In [10]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
Answer in Japanese.
"""

In [12]:
# TODO: Read image data, feed into Claude

with open("images/prop7.png", "rb") as f:
    image_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages, [
    {
        "type" : "image",
        "source": {
            "type" : "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type" : "text",
        "text" : prompt
    }
    ]
)

response = chat(
    messages
)

show_response(response)

────────────────────────────────────────────────────────────
  Model      : claude-sonnet-4-5-20250929
  Stop reason: end_turn
  Tokens     : input=2044, output=511
────────────────────────────────────────────────────────────
# 衛星画像の火災リスク分析

## 1. 住居の特定
主要住居は画像中央に位置する、複数の屋根面を持つL字型またはT字型の灰色の屋根構造物で、周囲を密集した樹木に囲まれており、最大の建築物として確認できます。

## 2. 樹木の張り出し分析
住居の屋根全体に複数の樹木の樹冠が直接覆い被さっており、屋根面積の推定75-100%が樹木の枝や葉によって覆われており、特に建物の北側と東側で密度の高い張り出しが見られます。

## 3. 火災リスク評価
張り出している樹木は屋根面との接触点が多数あり、飛び火が捕捉されやすい状況で、野火が発生した場合に樹木から建物への連続的な燃料経路が形成されています。

## 4. 防御可能空間の特定
住居周辺の樹木は完全に連続した樹冠を形成しており、地上植生から樹木、そして屋根へと火災が伝播する明確な燃料階梯(ファイヤーラダー)が複数箇所で確認されます。

## 5. 火災リスク評価
**火災リスク評価: 4（深刻なリスク）**

理由: 屋根面積の75%以上が樹木によって覆われており、住居が密集した植生に完全に囲まれ、防御可能空間がほぼ存在せず、複数の飛び火捕捉点と連続的な燃料経路が確認されるため。
────────────────────────────────────────────────────────────


### pdf解析の例

In [16]:
# TODO: Read image data, feed into Claude

with open("documents/earth.pdf", "rb") as f:
    file_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages, [
    {
        "type" : "document",
        "source": {
            "type" : "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    {
        "type" : "text",
        "text" : "Summarize the document in one sentence with Japanese."
    }
    ]
)

response = chat(
    messages,
    thinking=True,
)

show_response(response)

────────────────────────────────────────────────────────────
  Model      : claude-sonnet-4-5-20250929
  Stop reason: end_turn
  Tokens     : input=9657, output=244
────────────────────────────────────────────────────────────
地球は太陽から3番目の惑星で、生命が存在する唯一の天体であり、表面の70.8%を海洋が覆い、大気を持ち、約45億年前に形成され、月という1つの衛星を持つ太陽系で最も密度の高い岩石惑星です。
────────────────────────────────────────────────────────────
